# FINAL — Traffic Sign + Car License Plate Detection

Chạy **từ trên xuống** trên Google Colab. Trước khi chạy, chọn **Runtime → Change runtime type → T4 GPU**.

Pipeline cuối:
- **Biển báo giao thông Việt Nam**: YOLO + DIP color cue.
- **Biển số ô tô**: YOLO license-plate detector; `car / bus / truck` detector chỉ chạy nội bộ để lọc, **không vẽ box xe**.
- **Không còn** helmet, ROI, rider box hay Student ID.
- Video full-quality được lưu vào **Google Drive**; một bản preview nhẹ được tạo trong `/content` để xem ngay dưới cell.


## 1) Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Clone phiên bản cuối từ GitHub


In [ ]:
!rm -rf /content/DIP
!git clone -q -b feature/yolo-traffic-safety https://github.com/NVTruong473/DIP.git /content/DIP
%cd /content/DIP/END_DIP
!git log -1 --oneline


## 3) Install dependencies


In [ ]:
!pip install -q -r requirements.txt


## 4) Kiểm tra GPU + video đầu vào

Mặc định video nằm tại `MyDrive/DIP/video1.mp4`. Muốn thử video khác, chỉ cần đổi `VIDEO_NAME`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, torch

DRIVE_ROOT = Path('/content/drive/MyDrive/DIP')
VIDEO_NAME = 'video1.mp4'

VIDEO = DRIVE_ROOT / VIDEO_NAME
MODELS_DIR = DRIVE_ROOT / 'models'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'
STEM = VIDEO.stem

OUTPUT_VIDEO = OUTPUT_DIR / f'{STEM}_result.mp4'
OUTPUT_CSV = OUTPUT_DIR / f'{STEM}_result.csv'
PLATES_DIR = OUTPUT_DIR / f'{STEM}_plates'
TEMP_VIDEO = OUTPUT_DIR / f'{STEM}_result_temp.mp4'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('GPU        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('Input video:', VIDEO)
print('Exists     :', VIDEO.exists())

assert torch.cuda.is_available(), 'Hãy bật T4 GPU: Runtime → Change runtime type → T4 GPU'
assert VIDEO.exists(), f'Không tìm thấy video: {VIDEO}'

subprocess.run([
    'ffprobe', '-v', 'error',
    '-show_entries', 'format=duration:stream=width,height,r_frame_rate',
    '-of', 'default=noprint_wrappers=1',
    str(VIDEO)
], check=True)


## 5) Download/cache models

Chạy cell này mỗi lần cũng được. Các file đã có trong `MyDrive/DIP/models/` sẽ được Hugging Face cache/reuse.


In [ ]:
!python download_models.py --models-dir "/content/drive/MyDrive/DIP/models"


## 6) Xóa output cũ và chạy pipeline

Việc xóa output cũ giúp chắc chắn cell preview bên dưới luôn hiển thị **kết quả vừa chạy**, không phải video cũ trên Drive.


In [ ]:
for p in (OUTPUT_VIDEO, OUTPUT_CSV, TEMP_VIDEO):
    try:
        p.unlink()
    except FileNotFoundError:
        pass

shutil.rmtree(PLATES_DIR, ignore_errors=True)

cmd = [
    'python', 'main.py',
    '--input', str(VIDEO),
    '--output-dir', str(OUTPUT_DIR),
    '--models-dir', str(MODELS_DIR),
    '--sign-conf', '0.25',
    '--plate-conf', '0.30',
    '--vehicle-conf', '0.30',
    '--frame-stride', '1',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 7) Kiểm tra file kết quả trên Google Drive


In [ ]:
assert OUTPUT_VIDEO.exists() and OUTPUT_VIDEO.stat().st_size > 0, f'Không tạo được output: {OUTPUT_VIDEO}'
assert OUTPUT_CSV.exists(), f'Không tạo được CSV: {OUTPUT_CSV}'

print(f'Output video : {OUTPUT_VIDEO}')
print(f'Video size   : {OUTPUT_VIDEO.stat().st_size / 1024 / 1024:.1f} MB')
print(f'Detection CSV: {OUTPUT_CSV}')
print(f'Plate crops  : {PLATES_DIR}')
print(f'Crop count   : {len(list(PLATES_DIR.glob("*.jpg"))) if PLATES_DIR.exists() else 0}')

print('\nVideo codec check:')
subprocess.run([
    'ffprobe', '-v', 'error',
    '-select_streams', 'v:0',
    '-show_entries', 'stream=codec_name,pix_fmt,width,height',
    '-of', 'default=noprint_wrappers=1',
    str(OUTPUT_VIDEO)
], check=True)


## 8) Xem video kết quả ngay dưới cell

Bản full vẫn ở Google Drive. Cell này chỉ tạo một preview H.264 nhỏ hơn trong `/content` để player của Colab/Chrome chạy ổn định.


In [ ]:
from IPython.display import Video, display

PREVIEW = Path('/content') / f'{STEM}_result_preview.mp4'
try:
    PREVIEW.unlink()
except FileNotFoundError:
    pass

preview_cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-i', str(OUTPUT_VIDEO),
    '-vf', 'scale=854:-2',
    '-c:v', 'libx264',
    '-preset', 'veryfast',
    '-crf', '30',
    '-pix_fmt', 'yuv420p',
    '-tag:v', 'avc1',
    '-movflags', '+faststart',
    '-an',
    str(PREVIEW),
]
subprocess.run(preview_cmd, check=True)

assert PREVIEW.exists() and PREVIEW.stat().st_size > 0
print(f'Preview: {PREVIEW} ({PREVIEW.stat().st_size / 1024 / 1024:.1f} MB)')
display(Video(str(PREVIEW), embed=True, width=900, html_attributes='controls'))


## Hoàn tất

Bản cần giữ nằm tại:

`MyDrive/DIP/outputs/<tên-video>_result.mp4`

Với video mặc định:

`MyDrive/DIP/outputs/video1_result.mp4`

Các crop biển số tốt nhất nằm trong `MyDrive/DIP/outputs/video1_plates/`.

---

### Optional — giao diện upload video khác

Không cần chạy cell dưới đây cho bài demo mặc định. Chỉ chạy khi muốn upload/test một video khác bằng Gradio.


In [ ]:
# !python app.py
